In [7]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 41.7833,
	"longitude": 2.7333,
	"hourly": ["temperature_2m", "precipitation_probability", "precipitation", "rain", "weather_code", "pressure_msl", "visibility", "wind_speed_10m", "cloud_cover_low", "cloud_cover", "cloud_cover_mid", "cloud_cover_high", "uv_index", "uv_index_clear_sky", "is_day", "sunshine_duration"],
	"models": "best_match",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_rain = hourly.Variables(3).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(4).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(5).ValuesAsNumpy()
hourly_visibility = hourly.Variables(6).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(7).ValuesAsNumpy()
hourly_cloud_cover_low = hourly.Variables(8).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(9).ValuesAsNumpy()
hourly_cloud_cover_mid = hourly.Variables(10).ValuesAsNumpy()
hourly_cloud_cover_high = hourly.Variables(11).ValuesAsNumpy()
hourly_uv_index = hourly.Variables(12).ValuesAsNumpy()
hourly_uv_index_clear_sky = hourly.Variables(13).ValuesAsNumpy()
hourly_is_day = hourly.Variables(14).ValuesAsNumpy()
hourly_sunshine_duration = hourly.Variables(15).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["weather_code"] = hourly_weather_code
hourly_data["pressure_msl"] = hourly_pressure_msl
hourly_data["visibility"] = hourly_visibility
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
hourly_data["cloud_cover_high"] = hourly_cloud_cover_high
hourly_data["uv_index"] = hourly_uv_index
hourly_data["uv_index_clear_sky"] = hourly_uv_index_clear_sky
hourly_data["is_day"] = hourly_is_day
hourly_data["sunshine_duration"] = hourly_sunshine_duration

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 41.8125°N 2.75°E
Elevation: 107.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m  precipitation_probability  \
0   2026-03-21 00:00:00+00:00          3.7155                        0.0   
1   2026-03-21 01:00:00+00:00          3.1155                        0.0   
2   2026-03-21 02:00:00+00:00          2.5155                        0.0   
3   2026-03-21 03:00:00+00:00          1.3155                        0.0   
4   2026-03-21 04:00:00+00:00          0.7655                        0.0   
..                        ...             ...                        ...   
163 2026-03-27 19:00:00+00:00          6.3255                        4.0   
164 2026-03-27 20:00:00+00:00          4.1755                        4.0   
165 2026-03-27 21:00:00+00:00          2.2755                        5.0   
166 2026-03-27 22:00:00+00:00          0.7755                        5.0   
167 2026-03-27 23:00:00+00:00         -0.4745                   

In [4]:
# hourly_dataframe.to_csv("hourly_data.csv", index = False)

In [5]:
hourly_dataframe.describe()

,temperature_2m,precipitation_probability,precipitation,rain,weather_code,pressure_msl,visibility,wind_speed_10m,cloud_cover_low,cloud_cover,cloud_cover_mid,cloud_cover_high,uv_index,uv_index_clear_sky,is_day,sunshine_duration
count,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000,168.000000
mean,9.342970,8.523809,0.020238,0.008333,10.946428,1015.493286,25394.613281,7.607176,20.505953,41.461311,28.395834,4.815476,1.371428,1.585417,0.541667,1650.153809
std,5.060715,18.605326,0.095754,0.041552,20.315847,3.885022,13093.332031,6.450130,26.225187,34.416122,31.868725,12.495575,1.868455,2.048121,0.499750,1735.613892
min,-2.224500,0.000000,0.000000,0.000000,0.000000,1008.299988,420.000000,0.360000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.348000,0.000000,0.000000,0.000000,0.000000,1012.475006,19155.000000,2.144786,0.000000,4.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,9.645500,0.000000,0.000000,0.000000,1.000000,1014.712524,24140.000000,5.326719,3.500000,42.500000,16.000000,0.000000,0.175000,0.250000,1.000000,0.000000
75%,13.429250,6.250000,0.000000,0.000000,3.000000,1018.450012,38025.000000,12.334806,41.500000,72.500000,50.250000,0.000000,2.612500,3.650000,1.000000,3600.000000
max,19.165501,90.000000,0.900000,0.300000,80.000000,1022.599976,41960.000000,25.959969,100.000000,100.000000,100.000000,66.000000,5.650000,5.700000,1.000000,3600.000000
